# Augmentation Agent

### Estimated time: 10-15 minutes

This lab runs a DSPy-based augmentation agent that analyzes the gap between
structured graph data and unstructured documents, then proposes schema-level
enrichments for the Neo4j graph.

The agent uses a two-stage pipeline:

1. **Gap Analysis** — Queries the Supervisor Agent (from Lab 6) which coordinates
   Genie (structured data) and the Knowledge Assistant (documents) to identify
   mismatches between customer interests and portfolio holdings.

2. **DSPy Analysis** — Passes the gap analysis through four concurrent DSPy analyzers
   that produce typed, structured output:
   - **Investment Themes** — emerging themes from market research
   - **New Entities** — suggested new node types for the graph
   - **Missing Attributes** — attributes missing from existing nodes
   - **Implied Relationships** — relationships implied but not captured

The results are saved to the Unity Catalog volume for use in **8 - Graph Enrichment**,
which resolves these schema-level suggestions into concrete node-to-node proposals
and writes them back to Neo4j.

### Prerequisites

- Completed **6 - Supervisor Agent** with the Supervisor Agent deployed as an endpoint
- Cluster libraries: `dspy>=3.0.4`, `mlflow[databricks]>=3.1`, `databricks-openai`
- The `lab_7_augmentation_agent` package available on the cluster
  (install via `%pip install -e /path/to/graph-enrichment` or add to sys.path)

## Step 1: Configuration

Enter your Supervisor Agent endpoint name in the widget. You can find this by
clicking the **cloud icon** in the Supervisor Agent UI. Endpoint names use the
`mas-` prefix (Multi-Agent Supervisor).

In [ ]:
dbutils.widgets.text(
    "supervisor_endpoint", "",
    "Supervisor Agent Endpoint (e.g. mas-xxxxxxxx-endpoint)"
)

In [ ]:
%run ./Includes/config

In [ ]:
SUPERVISOR_ENDPOINT = dbutils.widgets.get("supervisor_endpoint")
if not SUPERVISOR_ENDPOINT:
    raise ValueError(
        "Please enter the Supervisor Agent endpoint name in the widget above.\n"
        "Find it by clicking the cloud icon in the Supervisor Agent UI."
    )

# Get volume path for saving results
_scope = CONFIG["secrets"]["scope_name"]
VOLUME_PATH = dbutils.secrets.get(scope=_scope, key="volume_path")

TEMPERATURE = 0.1
MAX_TOKENS = 4000

print("Configuration:")
print(f"  Supervisor Endpoint: {SUPERVISOR_ENDPOINT}")
print(f"  Volume Path:         {VOLUME_PATH}")
print(f"  Temperature:         {TEMPERATURE}")
print(f"  Max Tokens:          {MAX_TOKENS}")

## Step 2: Setup

Make the `lab_7_augmentation_agent` package importable and verify the Databricks connection.

In [ ]:
import sys
import os

# Add the repo root to sys.path so lab_7_augmentation_agent is importable.
# Adjust this path if your workspace layout differs.
_repo_root = os.path.dirname(os.path.dirname(os.path.abspath(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
).replace("/Workspace", "/Workspace")))
# Fallback: if running locally or the above doesn't resolve, try parent of labs/
if not os.path.isdir(os.path.join(_repo_root, "lab_7_augmentation_agent")):
    _repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f"[OK] Added to sys.path: {_repo_root}")

In [ ]:
from databricks.sdk import WorkspaceClient

print("Verifying Databricks connection...")
client = WorkspaceClient()
print(f"[OK] Connected to: {client.config.host}")

## Step 3: Configure DSPy

Configure DSPy with the Supervisor Agent endpoint. The `DatabricksResponsesLM`
adapter translates between DSPy's interface and the Databricks Responses API
format used by Supervisor Agent endpoints.

In [ ]:
from lab_7_augmentation_agent.dspy_modules.config import (
    configure_dspy,
    setup_mlflow_tracing,
)

lm = configure_dspy(
    model_name=SUPERVISOR_ENDPOINT,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

setup_mlflow_tracing()

## Step 4: Query Supervisor Agent for Gap Analysis

The Supervisor Agent coordinates Genie (structured data) and the Knowledge Assistant
(unstructured documents) to identify gaps between what customers want and what they have.

This is the core use case: James Anderson's profile mentions renewable energy interest,
but his portfolio contains only technology stocks.

In [ ]:
from lab_7_augmentation_agent.dspy_modules.supervisor_client import fetch_gap_analysis

gap_analysis = fetch_gap_analysis(SUPERVISOR_ENDPOINT)

print(f"\nGap analysis: {len(gap_analysis):,} characters")
print("\n" + "=" * 60)
print("PREVIEW (first 2000 chars)")
print("=" * 60)
print(gap_analysis[:2000])
if len(gap_analysis) > 2000:
    print(f"\n... ({len(gap_analysis) - 2000:,} more characters)")

## Step 5: Run DSPy Analyses

The `GraphAugmentationAnalyzer` runs all four analyses concurrently via `dspy.Parallel`.
Each analyzer uses `ChainOfThought` for step-by-step reasoning before producing
typed Pydantic output.

This typically takes 1-2 minutes wall-clock time.

In [ ]:
from lab_7_augmentation_agent.dspy_modules.analyzers import GraphAugmentationAnalyzer
from lab_7_augmentation_agent.utils import print_response_summary

print("Running all analyses with GraphAugmentationAnalyzer...")
print("(Uses dspy.Parallel — typically 1-2 minutes wall-clock)\n")

analyzer = GraphAugmentationAnalyzer()
full_response = analyzer(gap_analysis)

print_response_summary(full_response)

## Step 6: Save Results

Save the augmentation results as JSON to the Unity Catalog volume.
**8 - Graph Enrichment** loads this file to resolve the schema-level suggestions
into instance-level proposals and write them back to Neo4j.

In [ ]:
import json
import os

output_path = os.path.join(VOLUME_PATH, "augmentation_results.json")
response_dict = full_response.model_dump(mode="json")

with open(output_path, "w") as f:
    json.dump(response_dict, f, indent=2, default=str)

print(f"[OK] Results saved to: {output_path}")
print(f"     Total suggestions: {response_dict['total_suggestions']}")
print(f"     High confidence:   {response_dict['high_confidence_count']}")
print(f"     Suggested nodes:         {len(response_dict['all_suggested_nodes'])}")
print(f"     Suggested relationships: {len(response_dict['all_suggested_relationships'])}")
print(f"     Suggested attributes:    {len(response_dict['all_suggested_attributes'])}")

## Next Steps

The augmentation analysis is complete. Continue to **8 - Graph Enrichment** to:

1. Resolve these schema-level suggestions into concrete node-to-node proposals
2. Filter proposals by confidence (HIGH → approve, MEDIUM → flag, LOW → reject)
3. Write approved enrichments back to Neo4j with provenance metadata
4. Validate the enriched graph

After enrichment, re-run **4 - Neo4j to Lakehouse** to export the enriched graph
back to Delta tables — demonstrating the compounding enrichment loop described
in the paper.